
# M-Way ANOVA + Post-Hoc Tests with Decision Making

This notebook demonstrates how to perform a **multi-factor (M-way) ANOVA** in Python, followed by appropriate **post-hoc analysis** and automated **decision making** using p-values.

### Example
We analyze `Portfolio_Return` using three categorical factors:

- **Strategy:** Momentum, Mean Reversion, Breakout
- **Market:** Bull, Bear, Sideways
- **Risk:** Low, Medium, High

The notebook shows:

1. Data preparation
2. Exploratory summaries
3. M-way ANOVA using an OLS factorial model
4. Main-effect and interaction decisions
5. Post-hoc Tukey HSD for a significant main effect
6. Simple pairwise comparisons for a significant interaction
7. A reusable function for ANOVA decision making


In [ ]:

# Install packages if needed:
# %pip install pandas numpy scipy statsmodels seaborn matplotlib


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

ALPHA = 0.05



## 1. Create an example dataset

The synthetic data below are designed only for learning. In a real project, replace this section with your actual dataset.


In [ ]:

rng = np.random.default_rng(42)

strategies = ["Momentum", "Mean Reversion", "Breakout"]
markets = ["Bull", "Bear", "Sideways"]
risks = ["Low", "Medium", "High"]

# Balanced factorial design: 20 observations per Strategy × Market × Risk cell
rows = []

strategy_effect = {
    "Momentum": 0.80,
    "Mean Reversion": 0.20,
    "Breakout": 0.45
}

market_effect = {
    "Bull": 0.70,
    "Bear": -0.45,
    "Sideways": 0.00
}

risk_effect = {
    "Low": -0.05,
    "Medium": 0.05,
    "High": 0.10
}

# Strategy × Market interaction
interaction_effect = {
    ("Momentum", "Bull"): 0.70,
    ("Momentum", "Bear"): -0.40,
    ("Momentum", "Sideways"): -0.10,
    ("Mean Reversion", "Bull"): 0.00,
    ("Mean Reversion", "Bear"): 0.25,
    ("Mean Reversion", "Sideways"): 0.05,
    ("Breakout", "Bull"): 0.20,
    ("Breakout", "Bear"): -0.05,
    ("Breakout", "Sideways"): 0.00
}

for strategy in strategies:
    for market in markets:
        for risk in risks:
            for _ in range(20):
                mean_return = (
                    1.0
                    + strategy_effect[strategy]
                    + market_effect[market]
                    + risk_effect[risk]
                    + interaction_effect[(strategy, market)]
                )

                portfolio_return = rng.normal(mean_return, 0.90)

                rows.append([
                    strategy,
                    market,
                    risk,
                    portfolio_return
                ])

df = pd.DataFrame(
    rows,
    columns=["Strategy", "Market", "Risk", "Portfolio_Return"]
)

df.head()


In [ ]:

print("Shape:", df.shape)
print("\nFactor levels:")
for col in ["Strategy", "Market", "Risk"]:
    print(f"{col}: {df[col].unique().tolist()}")

print("\nMissing values:")
print(df.isna().sum())

print("\nGroup means:")
display(
    df.groupby(["Strategy", "Market", "Risk"])["Portfolio_Return"]
      .agg(["count", "mean", "std"])
      .round(4)
)


## 2. Exploratory visualization

In [ ]:

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df,
    x="Strategy",
    y="Portfolio_Return",
    hue="Market"
)
plt.axhline(df["Portfolio_Return"].mean(), linestyle="--", label="Grand mean")
plt.title("Portfolio Return by Strategy and Market")
plt.legend()
plt.tight_layout()
plt.show()



## 3. Fit the M-way ANOVA model

For three factors, the full factorial model is:

`Return ~ Strategy * Market * Risk`

The `*` includes:

- all three main effects
- all two-way interactions
- the three-way interaction

This is important: if you want to test interactions, do not use only `Strategy + Market + Risk`.


In [ ]:

model = smf.ols(
    "Portfolio_Return ~ C(Strategy) * C(Market) * C(Risk)",
    data=df
).fit()

anova_results = anova_lm(model, typ=2)
anova_results



### Type II vs Type III ANOVA

For balanced experimental data, Type II ANOVA is often convenient.

If your study is **unbalanced** and you need to test effects in a full factorial model while accounting for interactions, Type III sums of squares are often considered. In that case, use treatment/sum contrasts deliberately and interpret the results according to the model specification.

The code below shows Type III as an optional alternative.


In [ ]:

# Optional Type III ANOVA:
#
# from statsmodels.formula.api import ols
# model_type3 = ols(
#     "Portfolio_Return ~ C(Strategy, Sum) * C(Market, Sum) * C(Risk, Sum)",
#     data=df
# ).fit()
# anova_type3 = anova_lm(model_type3, typ=3)
# anova_type3



## 4. Decision making from the ANOVA table

### Rule

- If **p-value ≤ α**, reject H₀ → statistically significant effect.
- If **p-value > α**, fail to reject H₀ → insufficient evidence of an effect.

For an interaction, a significant p-value means the effect of one factor depends on the level of another factor.


In [ ]:

def add_anova_decision(anova_table, alpha=0.05):
    result = anova_table.copy()
    result["Decision"] = np.where(
        result["PR(>F)"] <= alpha,
        "Reject H0 - Significant",
        "Fail to reject H0 - Not significant"
    )
    result["Interpretation"] = np.where(
        result["PR(>F)"] <= alpha,
        "Evidence of an effect",
        "Insufficient evidence of an effect"
    )
    return result

decision_table = add_anova_decision(anova_results, ALPHA)
display(decision_table)


In [ ]:

print("ANOVA DECISION SUMMARY")
print("-" * 70)

for effect, row in anova_results.iterrows():
    if effect == "Residual":
        continue

    p_value = row["PR(>F)"]

    if p_value <= ALPHA:
        print(f"{effect}: SIGNIFICANT (p={p_value:.4g}) -> Reject H0")
    else:
        print(f"{effect}: NOT SIGNIFICANT (p={p_value:.4g}) -> Fail to reject H0")



## 5. Identify significant main effects

A significant ANOVA main effect tells us that **at least one level differs**, but it does not tell us which specific pairs differ.

For a significant main effect with 3 or more levels, a common follow-up is **Tukey HSD**.


In [ ]:

def tukey_posthoc(data, outcome, factor, alpha=0.05):
    tukey = pairwise_tukeyhsd(
        endog=data[outcome],
        groups=data[factor],
        alpha=alpha
    )

    tukey_df = pd.DataFrame(
        data=tukey._results_table.data[1:],
        columns=tukey._results_table.data[0]
    )

    tukey_df["Decision"] = np.where(
        tukey_df["reject"],
        "Reject H0 - Significant pair",
        "Fail to reject H0 - Not significant"
    )

    return tukey, tukey_df

# Example: post-hoc for Strategy
strategy_p = anova_results.loc["C(Strategy)", "PR(>F)"]

if strategy_p <= ALPHA:
    tukey_strategy, tukey_strategy_df = tukey_posthoc(
        df, "Portfolio_Return", "Strategy", ALPHA
    )
    display(tukey_strategy_df)
else:
    print(
        f"Strategy main effect is not significant (p={strategy_p:.4g}); "
        "Tukey HSD is not required for the main effect."
    )


In [ ]:

# Visualize Tukey HSD for Strategy when applicable
if strategy_p <= ALPHA:
    tukey_strategy.plot_simultaneous()
    plt.title("Tukey HSD: Strategy Pairwise Comparisons")
    plt.tight_layout()
    plt.show()



## 6. Why interaction changes the post-hoc strategy

Suppose **Strategy × Market** is significant.

Then the average Strategy effect across all markets can hide important differences.

For example:

> Momentum may outperform other strategies in Bull markets but not in Bear markets.

Therefore, investigate **simple effects** such as:

- Strategy comparisons within each Market
- Market comparisons within each Strategy

The following code performs Tukey HSD for Strategy separately within each Market.


In [ ]:

def strat_within_market_posthoc(data, alpha=0.05):
    outputs = {}

    for market in data["Market"].unique():
        subset = data[data["Market"] == market]

        tukey, tukey_df = tukey_posthoc(
            subset,
            "Portfolio_Return",
            "Strategy",
            alpha
        )

        outputs[market] = tukey_df

        print(f"\n===== Strategy comparisons within {market} market =====")
        display(tukey_df)

    return outputs

strategy_market_p = anova_results.loc[
    "C(Strategy):C(Market)", "PR(>F)"
]

if strategy_market_p <= ALPHA:
    simple_effect_results = strat_within_market_posthoc(df, ALPHA)
else:
    print(
        f"Strategy × Market interaction is not significant "
        f"(p={strategy_market_p:.4g}); simple-effect post-hoc analysis "
        "is not required by this decision rule."
    )



## 7. Optional: all significant main effects automatically

This function checks the ANOVA table and runs Tukey HSD for significant categorical main effects with more than two levels.

For significant interactions, it reports that simple-effects analysis should be considered instead of blindly applying an overall Tukey test.


In [ ]:

factor_columns = {
    "C(Strategy)": "Strategy",
    "C(Market)": "Market",
    "C(Risk)": "Risk"
}

def automatic_posthoc(data, anova_table, outcome, factor_map, alpha=0.05):
    results = {}

    for anova_term, factor in factor_map.items():
        if anova_term not in anova_table.index:
            continue

        p_value = anova_table.loc[anova_term, "PR(>F)"]
        n_levels = data[factor].nunique()

        print(f"\n{factor}: p={p_value:.4g}")

        if p_value <= alpha and n_levels > 2:
            print("Significant main effect -> running Tukey HSD.")
            _, tukey_df = tukey_posthoc(data, outcome, factor, alpha)
            results[factor] = tukey_df
            display(tukey_df)

        elif p_value <= alpha and n_levels == 2:
            print(
                "Significant main effect with two levels. "
                "No multi-group Tukey test is necessary."
            )

        else:
            print("Not significant -> no main-effect Tukey HSD.")

    return results

automatic_results = automatic_posthoc(
    df,
    anova_results,
    "Portfolio_Return",
    factor_columns,
    ALPHA
)



## 8. Overall decision report

This produces a concise report suitable for a data-science workflow.


In [ ]:

def print_decision_report(anova_table, alpha=0.05):
    print("=" * 80)
    print("M-WAY ANOVA DECISION REPORT")
    print("=" * 80)

    for effect, row in anova_table.iterrows():
        if effect == "Residual":
            continue

        p = row["PR(>F)"]

        if p <= alpha:
            decision = "SIGNIFICANT -> Reject H0"
        else:
            decision = "NOT SIGNIFICANT -> Fail to reject H0"

        print(f"{effect:<45} p={p:.6f} | {decision}")

    print("=" * 80)
    print(f"Significance level α = {alpha}")

print_decision_report(anova_results, ALPHA)



## 9. Assumption checks

ANOVA inference relies on assumptions. Important checks include:

1. **Independence** — observations should be independent according to the study design.
2. **Normality of residuals** — inspect residual plots and/or use a diagnostic test where appropriate.
3. **Homogeneity of variance** — residual variance should be reasonably similar across groups/cells.

For large datasets, formal tests can become overly sensitive. Combine statistical tests with graphical diagnostics and domain knowledge.


In [ ]:

residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.scatterplot(x=fitted, y=residuals, ax=axes[0])
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Residuals vs Fitted")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")

sm.qqplot(residuals, line="45", ax=axes[1])
axes[1].set_title("Q-Q Plot of Residuals")

plt.tight_layout()
plt.show()

print("Shapiro-Wilk p-value:",
      stats.shapiro(residuals).pvalue)


In [ ]:

# Levene's test across the full Strategy × Market × Risk cells
df["Cell"] = (
    df["Strategy"].astype(str) + " | " +
    df["Market"].astype(str) + " | " +
    df["Risk"].astype(str)
)

groups = [
    group["Portfolio_Return"].values
    for _, group in df.groupby("Cell")
]

levene_stat, levene_p = stats.levene(*groups, center="median")

print(f"Levene statistic = {levene_stat:.4f}")
print(f"Levene p-value   = {levene_p:.6f}")

if levene_p <= ALPHA:
    print("Decision: Evidence of unequal variances.")
else:
    print("Decision: Insufficient evidence of unequal variances.")



# Final decision workflow

### M-way ANOVA
- **p ≤ 0.05:** Reject H₀ → evidence of a main effect or interaction.
- **p > 0.05:** Fail to reject H₀ → insufficient evidence of an effect.

### Post-hoc analysis
- Significant main effect with 3+ levels → **Tukey HSD** is a common choice.
- Significant interaction → examine **simple effects / planned pairwise comparisons**, such as Strategy within each Market.
- Do not interpret significant main effects in isolation when a relevant interaction is significant.

### Important
A statistically significant result does not automatically imply practical or economic significance. For financial applications, also examine effect sizes, confidence intervals, economic magnitude, and out-of-sample validity.
